<a href="https://colab.research.google.com/github/nyp-sit/iti121-2025s2/blob/main/L11/embedding_with_sentence_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generate Embeddings with Sentence Transformers

In this exercise, we will use Sentence transformer library to load (sentence transformer) embedding models and explore it uses in various NLP tasks such as text similarity, text classification, and document retrieval.

[Sentence Transformers](https://sbert.net/) (a.k.a. SBERT) is the go-to Python module for accessing, using, and training state-of-the-art embedding and reranker models.  

We will explore different open source embedding models such as Qwen and Gemma embedding models.

### Load Model

Use the `sentence-transformers` libraries to create an instance of a model class with Qwen3-Embedding-0.6B.  You can read the model card [here](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B).

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "Qwen/Qwen3-Embedding-0.6B"
model = SentenceTransformer(model_id).to(device=device)

print(f"Device: {model.device}")
print(model)
print("Total number of parameters in the model:", sum([p.numel() for _, p in model.named_parameters()]))

## Generating Embedding

An embedding is a numerical representation of text, like a word or sentence, that captures its semantic meaning. Essentially, it's a list of numbers (a vector) that allows computers to understand the relationships and context of words.

Let's see how Qwen3-Embedding would process three different words `["apple", "banana", "car"]`.

Qwen3-Embedding has been trained on vast amounts of text and has learned the relationships between words and concepts.

In [ ]:
words = ["apple", "banana", "car"]

# Calculate embeddings by calling model.encode()
embeddings = model.encode(words)

print(embeddings)
for idx, embedding in enumerate(embeddings):
  print(f"Embedding {idx+1} (shape): {embedding.shape}")

In [ ]:
model.similarity(embeddings[0], embeddings[2])

The model outpus a numerical vector for each sentence. The actual vectors are very long (1024), but for simplicity, those are presented with a few dimensions.

The key isn't the individual numbers themselves, but **the distance between the vectors**. If we were to plot these vectors in a multi-dimensional space, The vectors for `apple` and `banana` would be very close to each other. And the vector for `car` would be far away from the other two.

## Determining Similarity

In this section, we use embeddings to determine how sementically similar different sentences are. Here we show examples with high, medieum, and low similarity scores.

- High Similarity:
  - Sentence A: "The chef prepared a delicious meal for the guests."
  - Sentence B: "A tasty dinner was cooked by the chef for the visitors."
  - Reasoning: Both sentences describe the same event using different words and grammatical structures (active vs. passive voice). They convey the same core meaning.

- Medium Similarity:
  - Sentence A: "She is an expert in machine learning."
  - Sentence B: "He has a deep interest in artificial intelligence."
  - Reasoning: The sentences are related as machine learning is a subfield of artificial intelligence. However, they talk about different people with different levels of engagement (expert vs. interest).

- Low Similarity:
  - Sentence A: "The weather in Tokyo is sunny today."
  - Sentence B: "I need to buy groceries for the week."
  - Reasoning: The two sentences are on completely unrelated topics and share no semantic overlap.

In [ ]:
# The sentences to encode
sentence_high = [
    "The chef prepared a delicious meal for the guests.",
    "A tasty dinner was cooked by the chef for the visitors."
]
sentence_medium = [
    "She is an expert in machine learning.",
    "He has a deep interest in artificial intelligence."
]
sentence_low = [
    "The weather in Tokyo is sunny today.",
    "I need to buy groceries for the week."
]

for sentence in [sentence_high, sentence_medium, sentence_low]:
  print(sentence)
  embeddings = model.encode(sentence)
  similarities = model.similarity(embeddings[0], embeddings[1])
  print("score: ", similarities.numpy()[0][0])

### Using Prompts

To generate the best embeddings, you should add an "instructional prompt" or "task" to the beginning of your input text. These prompts optimize the embeddings for specific tasks, such as document retrieval or question answering, and help the model distinguish between different input types, like a search query versus a document.

#### How to Apply Prompts

You can apply a prompt during inference in three ways.

1.  **Using the `prompt` argument**<br>
    Pass the full prompt string directly to the `encode` method. This gives you precise control.
    ```python
    embeddings = model.encode(
        sentence,
        prompt="task: sentence similarity | query: "
    )
    ```
2.  **Using the `prompt_name` argument**<br>
    Select a predefined prompt by its name. These prompts are loaded from the model's configuration or during its initialization.
    ```python
    embeddings = model.encode(sentence, prompt_name="STS")
    ```
3.  **Using the Default Prompt**<br>
    If you don't specify either `prompt` or `prompt_name`, the system will automatically use the prompt set as `default_prompt_name`, if no default is set, then no prompt is applied.
    ```python
    embeddings = model.encode(sentence)
    ```


In [ ]:
print("Available tasks:")
for name, prefix in model.prompts.items():
  print(f" {name}: \"{prefix}\"")
print("-"*80)

We can see that this embedding model supports predefined prompt_name `query`. Let's try out some queries:

In [ ]:
queries = [
    "What is the capital of China?",
    "Explain gravity",
]
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",
]

# Encode the queries and documents. Note that queries benefit from using a prompt
# Here we use the prompt called "query" stored under `model.prompts`, but you can
# also pass your own prompt via the `prompt` argument
query_embeddings = model.encode(queries, prompt_name="query")
document_embeddings = model.encode(documents)

# Compute the (cosine) similarity between the query and document embeddings
similarity = model.similarity(query_embeddings, document_embeddings)
print(similarity)

### Classification

Classification is the task of assigning a piece of text to one or more predefined categories or labels. It's one of the most fundamental tasks in Natural Language Processing (NLP).

A practical application of text classification is customer support ticket routing. This process automatically directs customer queries to the correct department, saving time and reducing manual work.


Let's try generate the embeddings for both labels and sentence to be classified.

In [ ]:
labels = ["Billing Issue", "Technical Support", "Sales Inquiry"]

sentence = [
  "Excuse me, the app freezes on the login screen. It won't work even when I try to reset my password.",
  "I would like to inquire about your enterprise plan pricing and features for a team of 50 people.",
]

# Calculate embeddings by calling model.encode()
label_embeddings = model.encode(labels)
embeddings = model.encode(sentence)

# Calculate the embedding similarities
similarities = model.similarity(embeddings, label_embeddings)
print(similarities)

idx = similarities.argmax(1)
print(idx)

for example in sentence:
  print(example, "-> ", labels[idx[sentence.index(example)]])

We notice that the similarities between labels and sentences are quite low.

We can use prefix for the text we want to encode, with description of the task we want.  This can be done using the `prompt` argument.  Basically what encode will do, is to prepend the text with the prompt.

In [ ]:
labels = ["Billing Issue", "Technical Support", "Sales Inquiry"]

sentence = [
  "Excuse me, the app freezes on the login screen. It won't work even when I try to reset my password.",
  "I would like to inquire about your enterprise plan pricing and features for a team of 50 people.",
]

# Calculate embeddings by calling model.encode()
label_embeddings = model.encode(labels, prompt='task: classification | label: ')
embeddings = model.encode(sentence, prompt='task: classification | text: ')

# Calculate the embedding similarities
similarities = model.similarity(embeddings, label_embeddings)
print(similarities)

idx = similarities.argmax(1)
print(idx)

for example in sentence:
  print(example, "-> ", labels[idx[sentence.index(example)]])


## Matryoshka Representation Learning (MRL)

Some embedding models such as Qwen3, leverages MRL to provide multiple embedding sizes from one model. It's a clever training method that creates a single, high-quality embedding where the most important information is concentrated at the beginning of the vector. You can check if a model supports MRL by refering to their model cards.

This means you can get a smaller but still very useful embedding by simply taking the first `N` dimensions of the full embedding. Using smaller, truncated embeddings is significantly cheaper to store and faster to process, but this efficiency comes at the cost of potential lower quality of embeddings. MRL gives you the power to choose the optimal balance between this speed and accuracy for your application's specific needs.

Let's use three words `["apple", "banana", "car"]` and create simplified embeddings to see how MRL works.

In [ ]:
def check_word_similarities():
  # Calculate the embedding similarities
  print("similarity function: ", model.similarity_fn_name)
  similarities = model.similarity(embeddings[0], embeddings[1:])
  print(similarities)

  for idx, word in enumerate(words[1:]):
    print("apple vs.", word, "-> score: ", similarities.numpy()[0][idx])

# Calculate embeddings by calling model.encode()
embeddings = model.encode(words)

check_word_similarities()

Now, for a faster application, you don't need a new model. Simply **truncate** the full embeddings to the first **512 dimensions**. For optimal results, it is also recommended to set `normalize_embeddings=True`, which scales the vectors to a unit length of 1.

In [ ]:
embeddings = model.encode(words, truncate_dim=512, normalize_embeddings=True)

for idx, embedding in enumerate(embeddings):
  print(f"Embedding {idx+1}: {embedding.shape}")

print("-"*80)
check_word_similarities()

In extremely constrained environments, you can further shorten the embeddings to just **128 dimensions**. You can also use the more efficient **dot-product** for similarity calculations instead of the standard **cosine** similarity.

In [ ]:
model = SentenceTransformer(model_id, truncate_dim=128, similarity_fn_name="dot").to(device=device)
embeddings = model.encode(words, normalize_embeddings=True)

for idx, embedding in enumerate(embeddings):
  print(f"Embedding {idx+1}: {embedding.shape}")

print("-"*80)
check_word_similarities()

## Simple RAG example
Retrieval is the task of finding the most relevant pieces of information from a large collection (a database, a set of documents, a website) based on the meaning of a query, not just keywords.

Imagine you work for a company, and you need to find information from the internal employee handbook, which is stored as a collection of hundreds of documents.

In [ ]:
corp_knowledge_base = [
  {
    "category": "HR & Leave Policies",
    "documents": [
      {
        "title": "Procedure for Unscheduled Absence",
        "content": "In the event of an illness or emergency preventing you from working, please notify both your direct manager and the HR department via email by 9:30 AM JST. The subject line should be 'Sick Leave - [Your Name]'. If the absence extends beyond two consecutive days, a doctor's certificate (診断書) will be required upon your return."
      },
      {
        "title": "Annual Leave Policy",
        "content": "Full-time employees are granted 10 days of annual paid leave in their first year. This leave is granted six months after the date of joining and increases each year based on length of service. For example, an employee in their third year of service is entitled to 14 days per year. For a detailed breakdown, please refer to the attached 'Annual Leave Accrual Table'."
      },
    ]
  },
  {
    "category": "IT & Security",
    "documents": [
      {
        "title": "Account Password Management",
        "content": "If you have forgotten your password or your account is locked, please use the self-service reset portal at https://reset.ourcompany. You will be prompted to answer your pre-configured security questions. For security reasons, the IT Help Desk cannot reset passwords over the phone or email. If you have not set up your security questions, please visit the IT support desk on the 12th floor of the Shibuya office with your employee ID card."
      },
      {
        "title": "Software Procurement Process",
        "content": "All requests for new software must be submitted through the 'IT Service Desk' portal under the 'Software Request' category. Please include a business justification for the request. All software licenses require approval from your department head before procurement can begin. Please note that standard productivity software is pre-approved and does not require this process."
      },
    ]
  },
  {
    "category": "Finance & Expenses",
    "documents": [
      {
        "title": "Expense Reimbursement Policy",
        "content": "To ensure timely processing, all expense claims for a given month must be submitted for approval no later than the 5th business day of the following month. For example, all expenses incurred in July must be submitted by the 5th business day of August. Submissions after this deadline may be processed in the next payment cycle."
      },
      {
        "title": "Business Trip Expense Guidelines",
        "content": "Travel expenses for business trips will, as a rule, be reimbursed based on the actual cost of the most logical and economical route. Please submit a travel expense application in advance when using the Shinkansen or airplanes. Taxis are permitted only when public transportation is unavailable or when transporting heavy equipment. Receipts are mandatory."
      },
    ]
  },
  {
    "category": "Office & Facilities",
    "documents": [
      {
        "title": "Conference Room Booking Instructions",
        "content": "All conference rooms in the Shibuya office can be reserved through your Calendar App. Create a new meeting invitation, add the attendees, and then use the 'Room Finder' feature to select an available room. Please be sure to select the correct floor. For meetings with more than 10 people, please book the 'Sakura' or 'Fuji' rooms on the 14th floor."
      },
      {
        "title": "Mail and Delivery Policy",
        "content": "The company's mail services are intended for business-related correspondence only. For security and liability reasons, employees are kindly requested to refrain from having personal parcels or mail delivered to the Shibuya office address. The front desk will not be able to accept or hold personal deliveries."
      },
    ]
  },
]


And imagine you have a question like below.

In [ ]:
question = "How do I reset my password?" # @param ["How many days of annual paid leave do I get?", "How do I reset my password?", "What travel expenses can be reimbursed for a business trip?", "Can I receive personal packages at the office?"] {type:"string", allow-input: true}

# Define a minimum confidence threshold for a match to be considered valid
similarity_threshold = 0.5 # @param {"type":"slider","min":0,"max":1,"step":0.1}

Search relevant document from the corporate knowledge base.

In [ ]:
# --- Helper Functions for Semantic Search ---

def _calculate_best_match(similarities):
    print(similarities)
    if similarities is None or similarities.nelement() == 0:
        return None, 0.0

    # Find the index and value of the highest score
    best_index = similarities.argmax().item()
    best_score = similarities[0, best_index].item()

    return best_index, best_score

def find_best_category(model, query, candidates):
    """
    Finds the most relevant category from a list of candidates.

    Args:
        model: The SentenceTransformer model.
        query: The user's query string.
        candidates: A list of category name strings.

    Returns:
        A tuple containing the index of the best category and its similarity score.
    """
    if not candidates:
        return None, 0.0

    # Encode the query and candidate categories for classification
    query_embedding = model.encode(query, prompt="task: classification | text: ")
    candidate_embeddings = model.encode(candidates, prompt="task: classification | label: ")

    print(candidates)
    return _calculate_best_match(model.similarity(query_embedding, candidate_embeddings))

def find_best_doc(model, query, candidates):
    """
    Finds the most relevant document from a list of candidates.

    Args:
        model: The SentenceTransformer model.
        query: The user's query string.
        candidates: A list of document dictionaries, each with 'title' and 'content'.

    Returns:
        A tuple containing the index of the best document and its similarity score.
    """
    if not candidates:
        return None, 0.0

    # Encode the query for retrieval
    query_embedding = model.encode(query, prompt="task: search result | query: ")

    # Encode the document for similarity check
    doc_texts = [
        f"title: {doc.get('title', 'none')} | text: {doc.get('content', '')}"
        for doc in candidates
    ]
    candidate_embeddings = model.encode(doc_texts)

    print([doc['title'] for doc in candidates])

    # Calculate cosine similarity
    return _calculate_best_match(model.similarity(query_embedding, candidate_embeddings))

In [ ]:
# Load the model

model = SentenceTransformer(model_id).to(device=device)

In [ ]:
# --- Main Search Logic ---

# In your application, `best_document` would result from a search.
# We initialize it to None to ensure it always exists.
best_document = None

# 1. Find the most relevant category
print("Step 1: Finding the best category...")
categories = [item["category"] for item in corp_knowledge_base]
best_category_index, category_score = find_best_category(
    model, question, categories
)

# Check if the category score meets the threshold
if category_score < similarity_threshold:
    print(f" `-> No relevant category found. The highest score was only {category_score:.2f}.")
else:
    best_category = corp_knowledge_base[best_category_index]
    print(f" `-> Category Found: '{best_category['category']}' (Score: {category_score:.2f})")

    # 2. Find the most relevant document ONLY if a good category was found
    print("\nStep 2: Finding the best document in that category...")
    best_document_index, document_score = find_best_doc(
        model, question, best_category["documents"]
    )

    # Check if the document score meets the threshold
    if document_score < similarity_threshold:
        print(f" `-> No relevant document found. The highest score was only {document_score:.2f}.")
    else:
        best_document = best_category["documents"][best_document_index]
        # 3. Display the final successful result
        print(f" `-> Document Found: '{best_document['title']}' (Score: {document_score:.2f})")


### Exercise

Try to run the examples above using another Embedding Model. One very lightweight and powerful model is EmbeddingGemma, which is a 300M parameter, state-of-the-art for its size, open embedding model from Google, built from Gemma 3.

#### Setup

Before starting this tutorial, complete the following steps:

* Get access to Gemma by logging into [Hugging Face](https://huggingface.co/google/embeddinggemma-300M) and selecting **Acknowledge license** for a Gemma model.
* Generate a Hugging Face [Access Token](https://huggingface.co/docs/hub/en/security-tokens#how-to-manage-user-access-token) and use it to login from Colab.

This notebook will run on either CPU or GPU.

After you have accepted the license, you need a valid Hugging Face Token to access the model.

In [ ]:
# Login into Hugging Face Hub
from huggingface_hub import login
login()